In [1]:
import nibabel as nib
import numpy as np
import random
import os
import pandas as pd

In [2]:
conductivity_ranges = {
        # Original tissues
        'White-Matter': [0.3, 0.36], 
        'Gray-Matter': [0.5, 0.6], 
        'CSF': [1.94, 2.328], 
        'Bone': [0.032, 0.0384],
        'Scalp': [0.45, 0.54], 
        'Eye_balls': [1.5, 1.8], 
        'Compact_bone': [0.01, 0.012], 
        'Spongy_bone': [0.041, 0.0492], 
        'Blood': [0.7, 0.84],
        'Muscle': [0.36, 0.432], 
        'Electrode': [1e6, 1e6], 
        'gel': [0.3, 0.36], 
        
        # Tumor regions
        'Necrosis': [1.5, 1.8], 
        'Edema': [1.5, 1.8], 
        'Enhanced_Tumor': [0.5, 0.6], 
        
    }

    # Define tumor label mapping (assuming tumor seg uses 1=Edema, 2=Enhanced, 3=Necrosis)
    #assign new labels starting from 1000 to avoid conflicts
tumor_mapping = {
    1: 1001,  # Necrosis
    2: 1002,  # Edema
    3: 1003   #Enhanced Tumor
}

 # Define all label mppings (original + tumor)
all_labels = {
    # Original labels (from your first lookup table)
    1: conductivity_ranges['White-Matter'],
    2: conductivity_ranges['Gray-Matter'],
    3: conductivity_ranges['CSF'],
    4: conductivity_ranges['Bone'],
    5: conductivity_ranges['Scalp'],
    6: conductivity_ranges['Eye_balls'],
    7: conductivity_ranges['Compact_bone'],
    8: conductivity_ranges['Spongy_bone'],
    9: conductivity_ranges['Blood'],
    10: conductivity_ranges['Muscle'],
    2000: conductivity_ranges['Electrode'],
    2400: conductivity_ranges['gel'],
    2001: conductivity_ranges['Electrode'],
    2401: conductivity_ranges['gel'],
    
    # Tumor labels
    tumor_mapping[1]: conductivity_ranges['Necrosis'],
    tumor_mapping[2]: conductivity_ranges['Edema'],
    tumor_mapping[3]: conductivity_ranges['Enhanced_Tumor']
}

voltage_mapping = {
    2000: -1,
    2001: 1
}

In [3]:
def merge_segmentations(base_path, tumor_path):
        base = nib.load(base_path)
        tumor = nib.load(tumor_path)
        
        # Verify dimensions match
        #if base.shape != tumor.shape:
            #raise ValueError("Segmentation dimensions don't match!")
        
        base_data = base.get_fdata()
        tumor_data = tumor.get_fdata()
        
        # Create merged segmentation (copy base first)
        merged = base_data.copy()
        
        
        
        # Replace tumor labels in merged data
        for old_label, new_label in tumor_mapping.items():
            merged[tumor_data == old_label] = new_label
        
        
        #merged_img = nib.Nifti1Image(merged, base.affine , base.header)
        #nib.save(merged_img, output_dir)
        return merged

In [4]:
# Step 3: Create conductivity maps
def create_conductivity_map(seg_data, output_path , base_seg):
        cond_data = np.zeros_like(seg_data)
        
       
        
        # Assign conductivities
        for label, (low, high) in all_labels.items():
            mask = seg_data == label
            cond_data[mask] = random.uniform(low, high)
        
        
        # Save conductivity map
        cond_img = nib.Nifti1Image(cond_data ,nib.load(base_seg).affine, nib.load(base_seg).header)
        nib.save(cond_img, output_path)
        
        return cond_data

In [5]:
def create_vlotage_map(base_seg_path, output_path):
    labeled_image = nib.load(base_seg_path)
    labeled_data = labeled_image.get_fdata()

    voltage_data = np.zeros_like(labeled_data)
    
    for label, voltage in voltage_mapping.items():
        mask = labeled_data == label
        voltage_data[mask] = voltage


    voltage_image = nib.Nifti1Image(voltage_data, labeled_image.affine , labeled_image.header)
    nib.save(voltage_image, output_path)

    return voltage_data

In [7]:
df = pd.read_csv("E:/Datasets/Stage 2/add electrode folder/All Subjects .csv")
subject_id_column = 'Subject ID'
for subject_id in df[subject_id_column]:
    # Define paths (change these to your actual file paths)
    tumor_seg_path = f"E:/Datasets/Re-Creation/Co_registration/{subject_id}/seg/seg_to_mni.nii.gz"
    output_base_dir = "D:/Results/extrapolation/conductivity"
    output_dir = os.path.join(output_base_dir, f"{subject_id}")
    
    voltage_output_base_dir = "D:/Results/extrapolation/voltage"
    voltage_output_dir = os.path.join(voltage_output_base_dir, f"{subject_id}")
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(voltage_output_dir, exist_ok=True)
    
    for i in range(3, 5):
        base_seg_path = f"E:/Datasets/Stage 2/labeled nifti with electrodes/{subject_id}/labels_{i}.nii.gz"
        # Perform the merge
        merged_data = merge_segmentations(base_seg_path, tumor_seg_path)
        # Generate 5 conductivity maps
        
        cond_map = create_conductivity_map(
            merged_data,
            os.path.join(output_dir, f'conductivity_map_{i}.nii.gz'),
            base_seg_path
        )
    
        voltage_map = create_vlotage_map(base_seg_path, 
            os.path.join(voltage_output_dir, f'voltage_map_{i}.nii.gz')
        )
        
    print(f"Created 2 conductivity maps in {output_dir}")

Created 2 conductivity maps in D:/Results/extrapolation/conductivity\TCGA-02-0006
Created 2 conductivity maps in D:/Results/extrapolation/conductivity\TCGA-02-0009
Created 2 conductivity maps in D:/Results/extrapolation/conductivity\TCGA-02-0011
Created 2 conductivity maps in D:/Results/extrapolation/conductivity\TCGA-02-0027
Created 2 conductivity maps in D:/Results/extrapolation/conductivity\TCGA-02-0033
Created 2 conductivity maps in D:/Results/extrapolation/conductivity\TCGA-02-0034
Created 2 conductivity maps in D:/Results/extrapolation/conductivity\TCGA-02-0037
Created 2 conductivity maps in D:/Results/extrapolation/conductivity\TCGA-02-0046
Created 2 conductivity maps in D:/Results/extrapolation/conductivity\TCGA-02-0047
Created 2 conductivity maps in D:/Results/extrapolation/conductivity\TCGA-02-0054
Created 2 conductivity maps in D:/Results/extrapolation/conductivity\TCGA-02-0059
Created 2 conductivity maps in D:/Results/extrapolation/conductivity\TCGA-02-0064
Created 2 conduc